# Outbound-call retry logic — RETRY-1..4, M3-26 interaction

Tests `server/utils.py`'s `call_with_retry()` — the shared retry wrapper added by the
retry-logic implementation spec (2026-08-14) — both as a standalone unit and wired into several
real outbound call sites (`server/hip_linking.py`, `server/auth.py`, `server/abha.py`,
`server/facility.py`, `server/hiu_consent.py`).

Uses the shared `harness.py` (storage isolation via `activate_scratch_storage()`, `FakeResponse`
for stubbed outbound responses) established in `set_a_idempotency_M2-9_M2-10_M2-11_M2-12_M2-13.ipynb`
and reused across every notebook in this suite. Outbound calls are stubbed by patching
`requests.post`/`requests.get` on the specific call-site module (matching this suite's existing
convention, e.g. `login_runner_and_cert_checks_*.ipynb`), and `server.utils.time.sleep` is patched
to a no-op recorder so these run instantly instead of actually waiting out the real 1s retry delay.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import requests as requests_module
from unittest.mock import patch

import harness
from server.utils import call_with_retry


repo root on sys.path: C:\Users\hp\Desktop\Aayush\repo


---
## RETRY-1 — a transient failure (ConnectionError) that clears on the 3rd attempt is reported as an overall success

**Unit-level check of `call_with_retry()` itself.** A `ConnectionError` on attempts 1 and 2, success on
attempt 3.

**Pass criteria:** exactly 3 attempts are made, the final result is the successful response, and exactly
2 retry delays of 1s each (flat, no backoff) were recorded via the patched `time.sleep`.

In [2]:
call_count = {"n": 0}

def flaky_make_request():
    call_count["n"] += 1
    if call_count["n"] < 3:
        raise requests_module.exceptions.ConnectionError("simulated network blip")
    return harness.FakeResponse(200, {"ok": True})

sleeps = []
with patch("server.utils.time.sleep", lambda s: sleeps.append(s)):
    response = call_with_retry(flaky_make_request, description="RETRY-1 unit test")

harness.check("exactly 3 attempts were made (2 transient failures + 1 success)", call_count["n"] == 3)
harness.check("the final result is the successful response", response.status_code == 200)
harness.check("exactly 2 retry delays of 1.0s each were recorded (flat, no backoff)", sleeps == [1.0, 1.0])


PASS -- exactly 3 attempts were made (2 transient failures + 1 success)
PASS -- the final result is the successful response
PASS -- exactly 2 retry delays of 1.0s each were recorded (flat, no backoff)


---
## RETRY-1b — same case, wired into a real call site (`server/hip_linking.py`'s `generate_link_token()`)

Confirms the wrapper is actually reachable end-to-end through a real outbound function, not just in
isolation: `requests.post` fails (ConnectionError) twice, then returns ABDM's real 202-Accepted shape on
the 3rd attempt. `get_gateway_token()` is stubbed out (no real gateway session needed for this test).

In [3]:
import server.hip_linking as hip_linking

harness.activate_scratch_storage("retry_1b")

call_count = {"n": 0}

def flaky_post(url, json=None, headers=None, timeout=None):
    call_count["n"] += 1
    if call_count["n"] < 3:
        raise requests_module.exceptions.ConnectionError("simulated network blip")
    return harness.FakeResponse(202, {})

sleeps = []
with patch.object(hip_linking, "get_gateway_token", lambda: "fake-gateway-token"), \
     patch.object(hip_linking.requests, "post", flaky_post), \
     patch("server.utils.time.sleep", lambda s: sleeps.append(s)):
    response = hip_linking.generate_link_token(
        hip_id="TEST-HIP",
        abha_address="retrytest@sbx",
        name="Retry Test Patient",
        gender="Male",
        year_of_birth=1990,
        abha_number="12-3456-7890-1234",
    )

harness.check("all 3 attempts were made through the real generate_link_token() call", call_count["n"] == 3)
harness.check("the final response is the successful 202", response.status_code == 202)
harness.check("exactly 2 retry delays of 1.0s each", sleeps == [1.0, 1.0])


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_retry_1b_t796w9l3
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- all 3 attempts were made through the real generate_link_token() call
PASS -- the final response is the successful 202
PASS -- exactly 2 retry delays of 1.0s each


---
## RETRY-2 — a call that fails all 3 attempts is reported as a failure only after all 3, not after the first

`server/auth.py`'s `find_bridge_service_by_id()` (a background/admin lookup) with `requests.get` rigged
to return a persistent `503` on every attempt.

**Pass criteria:** exactly 3 attempts are made (not 1), the final response still faithfully reports the
503 (not silently swallowed into a success), and exactly 2 retry delays were recorded.

In [4]:
import server.auth as auth

call_count = {"n": 0}

def always_fail_get(url, headers=None, timeout=None):
    call_count["n"] += 1
    return harness.FakeResponse(503, {"error": "Service Unavailable"})

sleeps = []
with patch.object(auth, "get_gateway_token", lambda: "fake-gateway-token"), \
     patch.object(auth.requests, "get", always_fail_get), \
     patch("server.utils.time.sleep", lambda s: sleeps.append(s)):
    response = auth.find_bridge_service_by_id(service_id="TestClinicHIP")

harness.check("all 3 attempts were made before giving up (not just 1)", call_count["n"] == 3)
harness.check("the persistent 503 is still faithfully reported, not silently swallowed", response.status_code == 503)
harness.check("exactly 2 retry delays of 1.0s each before giving up", sleeps == [1.0, 1.0])


PASS -- all 3 attempts were made before giving up (not just 1)
PASS -- the persistent 503 is still faithfully reported, not silently swallowed
PASS -- exactly 2 retry delays of 1.0s each before giving up


---
## RETRY-3 — a user-input-error response (wrong-OTP shape) is NOT retried, and still routes to the existing re-prompt behavior unchanged

`server/abha.py`'s `verify_otp()`, stubbed to return ABDM's real, confirmed wrong-OTP shape
(`200 OK` with `authResult: "failed"` — see `tools/m1_test_suite/login_runner.py`'s
`verify_login_otp()` docstring for where this shape was first confirmed live). This is a 200, not a
5xx/429/408, and `abha.py` never calls `raise_for_status()` -- so the default transient classifier never
sees it as retryable, and `verify_login_otp()`'s own re-prompt loop (unmodified by this pass) is the only
thing that ever decides what happens next.

**Pass criteria:** exactly 1 attempt is made (no retry), no retry delay is triggered, and the response
object handed back is the untouched wrong-OTP shape -- exactly what `verify_login_otp()` already knows
how to handle.

In [5]:
import server.abha as abha

call_count = {"n": 0}

def wrong_otp_post(url, headers=None, json=None, timeout=None):
    call_count["n"] += 1
    return harness.FakeResponse(200, {
        "txnId": "txn-retry-3",
        "authResult": "failed",
        "message": "Please enter a valid OTP. Entered OTP is either expired or incorrect.",
    })

sleeps = []
with patch.object(abha, "get_gateway_token", lambda: "fake-gateway-token"), \
     patch.object(abha.requests, "post", wrong_otp_post), \
     patch("server.utils.time.sleep", lambda s: sleeps.append(s)):
    response = abha.verify_otp(
        action="profile/login",
        scope=["abha-login", "aadhaar-verify"],
        txn_id="txn-retry-3",
        otp_value="encrypted-otp-placeholder",
    )

harness.check("the wrong-OTP response (200 + authResult=failed) was NOT retried -- exactly 1 attempt", call_count["n"] == 1)
harness.check("no retry delay was ever triggered", sleeps == [])
harness.check(
    "the response still carries the original wrong-OTP shape unchanged, for login_runner.py's own re-prompt logic",
    response.status_code == 200 and response.json()["authResult"] == "failed",
)


PASS -- the wrong-OTP response (200 + authResult=failed) was NOT retried -- exactly 1 attempt
PASS -- no retry delay was ever triggered
PASS -- the response still carries the original wrong-OTP shape unchanged, for login_runner.py's own re-prompt logic


---
## RETRY-4 — a category-3 "our own bug" response is NOT retried and NOT treated as a user-input error either

`server/facility.py`'s `register_bridge_service()`, stubbed to return a generic `400` that doesn't match
any documented user-input validation shape (this endpoint's own docstring already flags its
success/failure response shape as unconfirmed) -- the kind of response that usually means a bug in how
we built the request, not bad user input.

**Pass criteria:** exactly 1 attempt is made (a 400 is not in the transient set: not a connection error,
not a timeout, not 5xx/429/408), no retry delay is triggered, and the response is returned exactly as-is
for the caller's own existing `log_error()`-and-inspect path -- this wrapper doesn't need, and doesn't
attempt, to distinguish "user's fault" from "our fault" for a case it isn't retrying either way.

In [6]:
import server.facility as facility

call_count = {"n": 0}

def our_bug_post(url, json=None, headers=None, timeout=None):
    call_count["n"] += 1
    return harness.FakeResponse(400, {
        "error": {"code": "ABDM-9999", "message": "Some generic validation failure not matching any known user-input shape"}
    })

sleeps = []
with patch.object(facility, "get_gateway_token", lambda: "fake-gateway-token"), \
     patch.object(facility.requests, "post", our_bug_post), \
     patch("server.utils.time.sleep", lambda s: sleeps.append(s)):
    response = facility.register_bridge_service(
        facility_id="IN1234567890",
        facility_name="Retry Test Facility",
        hip_name="RETRY TEST",
        service_type="HIP",
    )

harness.check("a generic 400 ('our own bug' category) was NOT retried -- exactly 1 attempt", call_count["n"] == 1)
harness.check("NOT treated as transient -- no retry delay", sleeps == [])
harness.check("the response is returned as-is, unchanged, for the caller's own inspection", response.status_code == 400)


PASS -- a generic 400 ('our own bug' category) was NOT retried -- exactly 1 attempt
PASS -- NOT treated as transient -- no retry delay
PASS -- the response is returned as-is, unchanged, for the caller's own inspection


---
## M3-26 interaction — one artefact exhausting all 3 retries must not stop the others from being attempted and acked

**Why this matters:** M3-26 (see `consent_notify_resilience_checks_M3-26.ipynb`) fixed a real bug where
any single artefact's `fetch_consent()` failure, in a multi-hospital consent grant's per-artefact loop,
used to propagate out and silently drop the ack for every artefact in the notification -- fixed by
wrapping each artefact's `fetch_consent()` call in its own try/except
(`consent_hiu_notify_service.py`'s `process_consent_hiu_notify()`).

This retry wrapper sits INSIDE `fetch_consent()` (server/hiu_consent.py), fully self-contained -- so it
must compose cleanly with that per-artefact isolation: hospital 2's fetch exhausting its own full
3-attempt retry budget and still failing should behave exactly like hospital 2's fetch failing once used
to (an exception the per-artefact try/except catches and logs), while hospitals 1 and 3 -- whose own
fetches succeed on the first try -- must NOT be delayed or blocked by hospital 2's retries, and must still
get their own fetch attempted and included in the final ack.

**Pass criteria:** hospital 1 and hospital 3 are each attempted exactly once; hospital 2 is attempted the
full 3 times (its own retry budget, exhausted independently) before its exception is caught; ABDM still
receives one ack covering all 3 artefacts.

In [7]:
import server.hiu_consent as hiu_consent
import server.callbacks.services.consent_hiu_notify_service as consent_notify_service
from server.callbacks.repository.pending_consent_request_repository import (
    save_pending_consent_request, link_consent_request_id,
)

harness.activate_scratch_storage("retry_m3_26")

save_pending_consent_request("orig-req-retry-m326", {"hiu_id": "HIU-1"})
link_consent_request_id("orig-req-retry-m326", "consent-req-retry-m326")

fetch_attempts = {"consent-hospital-1": 0, "consent-hospital-2": 0, "consent-hospital-3": 0}
ack_calls = []

def fake_post(url, json=None, headers=None, timeout=None):
    if url.endswith("/consent/v3/fetch"):
        consent_id = json["consentId"]
        fetch_attempts[consent_id] += 1
        if consent_id == "consent-hospital-2":
            # Persistently transient -- every attempt fails, so this exhausts
            # the full 3-attempt retry budget before fetch_consent() finally
            # raises for real.
            raise requests_module.exceptions.ConnectionError("simulated persistent network failure for hospital 2")
        return harness.FakeResponse(202, {})
    elif url.endswith("/consent/v3/request/hiu/on-notify"):
        ack_calls.append(json["acknowledgement"])
        return harness.FakeResponse(202, {})
    raise AssertionError(f"unexpected URL in test: {url}")

callback_data = {
    "headers": {"request-id": "req-retry-m326"},
    "body": {"notification": {
        "consentRequestId": "consent-req-retry-m326",
        "status": "GRANTED",
        "consentArtefacts": [
            {"id": "consent-hospital-1"},
            {"id": "consent-hospital-2"},  # this one exhausts all 3 retry attempts
            {"id": "consent-hospital-3"},
        ],
    }},
}

sleeps = []
with patch.object(hiu_consent, "get_gateway_token", lambda: "fake-gateway-token"), \
     patch.object(hiu_consent.requests, "post", fake_post), \
     patch("server.utils.time.sleep", lambda s: sleeps.append(s)):
    await consent_notify_service.process_consent_hiu_notify(callback_data)

harness.check("hospital 1's fetch succeeded on the first attempt (no retry needed)", fetch_attempts["consent-hospital-1"] == 1)
harness.check("hospital 2's fetch used its full 3-attempt retry budget before giving up", fetch_attempts["consent-hospital-2"] == 3)
harness.check("hospital 3's fetch succeeded on the first attempt -- NOT delayed or blocked by hospital 2's retries", fetch_attempts["consent-hospital-3"] == 1)
harness.check(
    "ABDM still received one ack covering all 3 artefacts, despite hospital 2 exhausting its retries and failing",
    len(ack_calls) == 1 and len(ack_calls[0]) == 3,
)
harness.check("hospital 2's own retries used its 2-delay budget (1s flat, no backoff)", sleeps == [1.0, 1.0])


[harness] scratch storage active at: C:\Users\hp\AppData\Local\Temp\edge_case_scratch_retry_m3_26_0zckfs_u
[harness] (NOT the repo's real storage/ directory -- nothing here touches that)
PASS -- hospital 1's fetch succeeded on the first attempt (no retry needed)
PASS -- hospital 2's fetch used its full 3-attempt retry budget before giving up
PASS -- hospital 3's fetch succeeded on the first attempt -- NOT delayed or blocked by hospital 2's retries
PASS -- ABDM still received one ack covering all 3 artefacts, despite hospital 2 exhausting its retries and failing
PASS -- hospital 2's own retries used its 2-delay budget (1s flat, no backoff)


---
## Summary

All checks above PASS against the real, wrapped service code:

- **RETRY-1 / RETRY-1b** -- a transient failure (ConnectionError) clearing on the 3rd attempt is reported
  as an overall success, both at the `call_with_retry()` unit level and through a real call site
  (`generate_link_token()`), with exactly 2 flat 1s retry delays.
- **RETRY-2** -- a persistently failing call (503 on every attempt) is only reported as a failure after
  all 3 attempts, not after the first.
- **RETRY-3** -- a user-input-error response (wrong-OTP shape) is never retried and reaches the caller
  completely unchanged, so `login_runner.py`'s existing re-prompt logic is untouched.
- **RETRY-4** -- a category-3 "our own bug" response (generic 400) is never retried and never
  misclassified as a user-input error.
- **M3-26 interaction** -- one artefact exhausting its full retry budget and still failing does not
  delay, block, or drop the other artefacts in the same multi-hospital consent grant; the ack still
  covers all of them. The new retry wrapper composes cleanly with the existing per-artefact error
  isolation instead of reintroducing the M3-26 bug.